
# TXT → SwDA CSV converter

Parses a plain text file where each line has the pattern:

- `Speaker|Utterance|Label` (e.g., `A|Okay.|b`)  
- or `|Utterance|Label` when speaker is omitted (leading `|`)

Outputs:
- `swda_all.csv` with columns: **text**, **label**, **speaker** (optional)
- optionally `swda_train.csv` and `swda_val.csv` via a **stratified split**

> After generating `swda_train.csv` / `swda_val.csv`, you can plug them directly into your **supervised DA** notebook.


In [ ]:

# If needed, uncomment to install dependencies
# %pip install -q pandas scikit-learn


In [ ]:

import re
import pandas as pd
from collections import Counter
from typing import List, Tuple, Dict, Optional

# === Edit this path to your TXT file ===
TXT_PATH = "full_set.txt"   # e.g., "/content/your_file.txt"

# Output files
OUT_ALL_CSV = "swda_all.csv"
OUT_TRAIN   = "swda_train.csv"
OUT_VAL     = "swda_val.csv"

# Perform stratified split?
DO_SPLIT = True
VAL_SIZE = 0.1
SEED     = 42

# 41 SwDA codes for validation
VALID_CODES = {
    "sd","b","sv","%","aa","ba","qy","ny","fc","qw","nn","bk","h","qy^d","bh","^q","bf",
    "fo_o_fw_\"_by_bc","na","ad","^2","b^m","qo","qh","^h","ar","ng","br","no","fp","qrr",
    "arp_nd","t3","oo_co_cc","aap_am","t1","bd","^g","qw^d","fa","ft"
}
print("Expecting labels in:", len(VALID_CODES), "codes")


In [ ]:

def parse_line(line: str) -> Optional[Tuple[Optional[str], str, str]]:
    """Parse a single line in the robust way:
    - label is everything after the **last** '|'
    - speaker is everything **before** the first '|'
    - utterance is in between
    Returns (speaker, text, label) or None if the line is not parseable.
    """
    s = line.rstrip('\n').strip()
    if not s or s.startswith("#"):
        return None  # ignore empty/comments

    # Ignore placeholder like '.... mais linhas abaixo'
    if s.startswith("...."):
        return None

    # Require at least two '|' separators overall
    if '|' not in s:
        return None
    if s.count('|') < 2:
        # handle a weird edge: prepend an empty speaker if only one pipe (rare)
        s = "|" + s

    # Split by last '|' to isolate label (labels never contain '|')
    try:
        head, label = s.rsplit("|", 1)
    except ValueError:
        return None

    label = label.strip()
    # Speaker is before first '|', utterance in between
    if "|" in head:
        speaker, text = head.split("|", 1)
    else:
        # if no first pipe, treat as no speaker
        speaker, text = "", head

    speaker = speaker.strip() or None
    text = text.strip()

    # Basic sanity checks
    if not text:
        return None
    if not label:
        return None

    return (speaker, text, label)


In [ ]:

# Load and parse the TXT
rows = []
bad_label_rows = []
with open(TXT_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        res = parse_line(line)
        if res is None:
            continue
        speaker, text, label = res
        if label not in VALID_CODES:
            bad_label_rows.append((i, speaker, text, label))
        rows.append({"speaker": speaker, "text": text, "label": label})

print(f"Parsed rows: {len(rows)}")
if bad_label_rows:
    print(f"⚠️ {len(bad_label_rows)} rows have labels outside SwDA-41. Showing first 5:")
    for r in bad_label_rows[:5]:
        print("  line", r[0], "| speaker:", r[1], "| label:", r[3], "| text:", r[2][:60], "...")
else:
    print("All labels are within the 41 SwDA codes.")


In [ ]:

# Create DataFrame and save all CSV
df = pd.DataFrame(rows)
# Ensure columns order
df = df[["text","label","speaker"]]
df.to_csv(OUT_ALL_CSV, index=False, encoding="utf-8")
print("Saved:", OUT_ALL_CSV)

# Show quick stats
print("\nLabel distribution (top 15):")
print(df["label"].value_counts().head(15))

print("\nSample rows:")
display(df.head(10))


In [ ]:

# Optional: stratified split into train/val CSVs
if DO_SPLIT:
    from sklearn.model_selection import StratifiedShuffleSplit
    df_clean = df.dropna(subset=["text","label"]).copy()
    y = df_clean["label"].values
    sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
    (train_idx, val_idx), = sss.split(pd.Series([0]*len(y)), y)

    df_train = df_clean.iloc[train_idx].reset_index(drop=True)
    df_val   = df_clean.iloc[val_idx].reset_index(drop=True)

    df_train.to_csv(OUT_TRAIN, index=False, encoding="utf-8")
    df_val.to_csv(OUT_VAL, index=False, encoding="utf-8")
    print(f"Saved stratified splits: {OUT_TRAIN} ({len(df_train)}), {OUT_VAL} ({len(df_val)})")
